# TP3 — PINN Resolution
## Temperature monitoring through Physics-Informed Neural Networks

This notebook implements the resolution of **Test Problem 3 (TP3)** via **PINNs**.  
The objective is to exploit **Physics-Informed Neural Networks (PINNs)** and simulated IoT-like measurements.

This stage corresponds to the **Direct Problem Submodule** of the Inference Engine described in the paper.

In [ ]:
import sys
from pyprojroot import here

PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

In [ ]:
from paths import DRT_PATH, TP3_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + DRT_PATH + TP3_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"
load_data_bound = ABS_PATH + "./files/data.csv"
load_initial_cond = ABS_PATH + "./files/tempdata.csv"
load_collocation_points = ABS_PATH + "files/input_int.csv"

fig_dir = "figures/"
loss_fig = ABS_PATH + fig_dir + "loss_data.png"

file_path = "rock.xdmf"

model_name_features = "./models/file"
model_saved_name = './models/model_weights'

Import of packages

In [ ]:
import json
import torch
import pandas as pd
import fenics as fe
import matplotlib.pyplot as plt

from model_acquisition import Blend2Pina, Msh2Xdmf

from pina.equation import Equation
from pina.solvers.pinns import PINN
from pina.condition import Condition
from pina.callbacks import MetricTracker
from pina.operators import grad, laplacian
from pina.model import ResidualFeedForward
from pina import LabelTensor, Trainer, Plotter
from pina.problem import SpatialProblem, TimeDependentProblem
from pytorch_lightning.callbacks import StochasticWeightAveraging

Set the double precision

In [ ]:
torch.set_default_dtype(torch.float64)
torch.set_float32_matmul_precision('high')

Definition of useful variables

In [ ]:
collocation_points = 400
boundary_points = 9_900

# Network variables
lear_rate = 5e-4
decay_rt = 1e-8

# Solver variables
epochs = 30_000
acc_str = 'gpu'
batch_dim = None
num_layers = 2
num_neurons = 200
input_neurons = 4
output_neurons = 1

## Geometry management and loading of data

Here the notebook focuses on the interaction between the preprocessed Blender 3D model (See notebook *00_ProblemSettings.ipynb*) with the PINA Geometry module.
In addition. in this fase the simulated data is loaded.

In [ ]:
filename = LOAD_MODEL + model_name
rock = Blend2Pina(filename)

rock_int = rock.intern(time_interval=[0,1])

In [ ]:
df = pd.read_csv(load_data_bound, sep=";", index_col=None)

df = df.sample(boundary_points)
input_pts = df.iloc[:, :4].values
input_pts = LabelTensor(
    x=torch.tensor(input_pts, dtype=torch.float64),
    labels=['x', 'y', 'z', 't']
)
output_pts = df.iloc[:, 4:].values
output_pts = LabelTensor(
    x=torch.reshape(torch.tensor(output_pts, dtype=torch.float64), (output_pts.shape[0], 1)),
    labels=['u']
)

In [ ]:
df_temp = pd.read_csv(load_initial_cond, sep=";", index_col=None)
t_initial = torch.tensor(df_temp.iloc[0, 1])

## Governing Physical Model

We consider the the following differential problem on the 3D domain $ \Omega \subset \mathbb{R}^3 $:

\begin{cases}
    u_t (x, y, z, t ) - \Delta u (x, y, z, t ) = 0 & \Omega \times [0, 1]
    \tag{1}
\end{cases}
with boundary conditions described by data generated below and initial condition that fixes a a constant temperature of the domain.

In the next block, the problem is defined.

In [ ]:
class Heat(SpatialProblem, TimeDependentProblem):

    input_variables = ['x','y','z','t']
    output_variables = ['u']
    spatial_domain = rock_int.spatial_domain
    temporal_domain = rock_int.temporal_domain

    def fix_parab_equation(input_, output_):
        u_t = grad(output_, input_, components=['u'], d=['t'])
        my_lapl = laplacian(output_, input_, components=['u'], d=['x', 'y', 'z'])
        return u_t - my_lapl

    conditions = {
        'Gamma' : Condition(
            input_points=input_pts,
            output_points=output_pts
        ),
        'Omega' : Condition(
            location=rock_int,
            equation=Equation(fix_parab_equation)
        )
    }

    conditions['Gamma'].data_weight = .33
    conditions['Omega'].data_weight = .77

After the definition of the class, the sampling of collocation and boundary points is performed.

In [ ]:
problem = Heat()
print(problem.input_variables)
print(problem.output_variables)

In [ ]:
try:
    df = pd.read_csv(load_collocation_points, sep=";", index_col=0)
    problem.discretise_domain(
        1,
        locations=["Omega"]
    )
    problem.input_pts["Omega"] = LabelTensor(
        torch.tensor(df.sample(collocation_points).values),
        labels=['x', 'y', 'z', 't']
    )

except:
    problem.discretise_domain(
        collocation_points,
        locations=["Omega"]
    )
    df = pd.DataFrame(
        problem.input_pts["Omega"].tensor.detach().numpy()
    )
    df.to_csv(load_collocation_points, sep = ";")

## Physics-Informed Neural Network (PINN)

A Physics-Informed Neural Network is employed to approximate the solution

\begin{equation}
u(x,y,z) \approx u_{\theta}(x,y,z),
\end{equation}

where $ u_{\theta} $ is a neural network parameterized by weights $ \theta $.

### PINN Loss Function

The training of the PINN is driven by a composite loss function:

\begin{equation}
\mathcal{L} =
\mathcal{L}_{\text{PDE}} +
\mathcal{L}_{\text{data}},
\end{equation}

where:

- **PDE residual loss**

\begin{equation}
\mathcal{L}_{\text{PDE}} =
\frac{1}{N_{\Omega}}
\sum_{i=1}^{N_{\Omega}}
\left\|
\Delta u_{\theta}(x_i) + (\alpha^2 + \beta^2)\pi^2 \lambda x_i \cos(\alpha \pi y_i)\sin(\beta \pi z_i)
\right\|^2,
\end{equation}

- **Data loss**

\begin{equation}
\mathcal{L}_{\text{data}} =
\frac{1}{N_d}
\sum_{i=1}^{N_d}
\left\|
u_{\theta}(x_i) - u_i
\right\|^2.
\end{equation}

The initial condition is satisfied by hard constraints in the NN model.

In [ ]:
class HardMLP(torch.nn.Module):

    def __init__(self, t_initial, *args, **kwargs):
        super().__init__()
        self.t_initial = t_initial

        self.layers = ResidualFeedForward(*args, **kwargs)

    def forward(self, x):
        return self.t_initial + x.extract('t')*self.layers(x)

Inizialization of the NN model, the PINN solver, the Trainer. Then, the training phase starts.

In [ ]:
# Model
model = HardMLP(t_initial, input_dimensions=input_neurons,output_dimensions=output_neurons,n_layers=num_layers,inner_size=num_neurons)
# Solver
pinn=PINN(
    problem=problem,
    model=model,
    optimizer_kwargs={
        'lr' : lear_rate,
        'weight_decay' : decay_rt
    },
)

# Trainer
trainer=Trainer(
    solver=pinn,
    max_epochs=epochs,
    batch_size=batch_dim,
    accelerator=acc_str,
    precision='64-true',
    callbacks=[MetricTracker(), StochasticWeightAveraging(swa_lrs=5e-4, swa_epoch_start=.9)]
)

# Training phase
trainer.train()

Plot of the losses related to the training process.

In [ ]:
my_pl = Plotter()
my_pl.plot_loss(
    trainer=trainer,
    metrics=['Omega_loss'],
    label='Omega',
    logy=True
)
my_pl.plot_loss(
    trainer=trainer,
    metrics=['Gamma_loss'],
    label='Gamma',
    logy=True
)

plt.savefig(loss_fig, transparent=True)
plt.show()

Load of the mesh and prediciton on mesh points

In [ ]:
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

mesh_points = LabelTensor(
    mesh.coordinates(),
    labels=['x', 'y', 'z']
)

In [ ]:
time = torch.linspace(0, 1, 21)

time_rep = time.repeat_interleave(mesh_points.size(0)).unsqueeze(1)
mesh_points_rep = mesh_points.repeat(time.numel(), 1)

sol_points = LabelTensor(torch.cat([mesh_points_rep, time_rep], dim=1), ['x', 'y', 'z', 't'])

In [ ]:
pred = pinn.neural_net(sol_points)
pred = pred.reshape(pred.shape[0] // len(time), len(time))
pred = pred.T

## Export of solution and saving model

In [ ]:
column_xdmf = Msh2Xdmf("rock.msh", "rock")

column_xdmf.reset_files("_sol_pinn")
column_xdmf.add_solution(pred.tensor.detach().numpy(), "_sol_pinn", "solution_pinn", len(time))

In [ ]:
torch.save(model.state_dict(), model_saved_name + '.pth')

In [ ]:
create_model = {
    'num_hidden_layers' : num_layers,
    'num_hidden_neurons' : num_neurons,
    'input_dim' : input_neurons,
    'output_dim' : output_neurons
}

with open(model_name_features + "_data" + '.json', 'w') as file:
    json.dump(create_model, file, indent=4)